# Inspect One Transformer Batch - v11

This notebook loads one provider-built 1m-bar batch, checks tensor shapes and finite values, loads a model checkpoint if provided, runs one inference, and computes one Smooth L1 loss.

In [ ]:
from pathlib import Path

MODEL_VERSION = "v11"
MODEL_ROOT = Path(r"D:\\TradingData\\quant-research-workbench\\market_data\\models\\inhouse_transformer") / MODEL_VERSION

# Edit these values for the batch/checkpoint you want to inspect.
WORKSPACE = Path(r"D:\\TradingCodes\\quant-research-workbench")
SESSION_DATE = "2024-01-22"
TICKERS = ("USO",)  # Use a small tuple first. Example: ("USO", "CADL")
BATCH_SIZE = 16
CONTEXT_LENGTH = 64
HORIZON = 1
TARGET_COLUMNS = ("close",)
TARGET_MODE = "return_bps"
SESSION_SCOPE = "all"
DEVICE = "cuda"  # Use "cpu" if CUDA is unavailable.
USE_AMP = False  # Keep False while checking exact tensors/losses.
LOAD_CHECKPOINT_CONFIG = True  # Use checkpoint model/data shape when CHECKPOINT_PATH is set.

# Paste a checkpoint path, or leave empty to auto-load the latest v11 last.pt/best.pt under MODEL_ROOT.
CHECKPOINT_PATH = ""
# Example:
# CHECKPOINT_PATH = str(MODEL_ROOT / "feature_temporal_v11_return_bps_ctx64_h1_..." / "last.pt")

In [ ]:
import sys
import math
from dataclasses import fields
import numpy as np
import polars as pl
import torch

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.inhouse_transformer.v11.config import DataConfig, ModelConfig, ExperimentConfig
from research.inhouse_transformer.v11.data import RollingBarWindowDataset, available_sessions, target_values_to_bps
from research.inhouse_transformer.v11.model import FeatureTemporalTransformer, forecast_loss

np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5, sci_mode=False)

def dataclass_from_dict(cls, payload, tuple_fields=()):
    allowed = {field.name for field in fields(cls)}
    values = {key: value for key, value in payload.items() if key in allowed}
    if cls is DataConfig and "processed_root" in values:
        values["processed_root"] = Path(values["processed_root"])
    for name in tuple_fields:
        if name in values:
            values[name] = tuple(values[name])
    return cls(**values)

def latest_checkpoint(model_root):
    candidates = list(model_root.glob("*/last.pt")) + list(model_root.glob("*/best.pt"))
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None

checkpoint = None
checkpoint_path = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else latest_checkpoint(MODEL_ROOT)
if checkpoint_path:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    checkpoint_config = checkpoint.get("config", {}) if isinstance(checkpoint, dict) else {}
else:
    checkpoint_config = {}
checkpoint_version = checkpoint_config.get("experiment_version") if checkpoint_config else None
if checkpoint_version and checkpoint_version != MODEL_VERSION:
    raise ValueError(f"Checkpoint version {checkpoint_version!r} does not match notebook version {MODEL_VERSION!r}.")

if checkpoint_config and LOAD_CHECKPOINT_CONFIG:
    data_config = dataclass_from_dict(
        DataConfig,
        checkpoint_config.get("data", {}),
        tuple_fields=("target_columns", "input_feature_columns", "time_feature_columns", "tickers"),
    )
    model_config = dataclass_from_dict(ModelConfig, checkpoint_config.get("model", {}))
    print("using data/model shape from checkpoint config")
else:
    data_config = DataConfig(
        context_length=CONTEXT_LENGTH,
        horizon=HORIZON,
        target_mode=TARGET_MODE,
        target_columns=TARGET_COLUMNS,
        input_normalization="window_zscore_only",
    )
    model_config = ModelConfig()

# These are inspection controls, so they intentionally override checkpoint split/ticker settings.
data_config.train_start_date = SESSION_DATE
data_config.train_end_date = SESSION_DATE
data_config.validation_start_date = SESSION_DATE
data_config.validation_end_date = SESSION_DATE
data_config.test_start_date = SESSION_DATE
data_config.test_end_date = SESSION_DATE
data_config.session_scope = SESSION_SCOPE
data_config.tickers = TICKERS
data_config.max_tickers = len(TICKERS)
experiment_config = ExperimentConfig(data=data_config, model=model_config)

device = torch.device(DEVICE if DEVICE == "cpu" or torch.cuda.is_available() else "cpu")
print(f"device={device}")
print(f"input_features={len(data_config.input_feature_columns)} {data_config.input_feature_columns}")
print(f"context={data_config.context_length} horizon={data_config.horizon} targets={data_config.target_columns}")
print(f"target_mode={data_config.target_mode} input_normalization={data_config.input_normalization}")
print(f"time_features={len(data_config.time_feature_columns)} {data_config.time_feature_columns}")
print(f"targets={data_config.target_columns} horizon={data_config.horizon} target_mode={data_config.target_mode}")
print(f"input_normalization={data_config.input_normalization}")

In [ ]:
sessions = available_sessions(data_config.processed_root, SESSION_DATE, SESSION_DATE)
dataset = RollingBarWindowDataset(
    config=data_config,
    sessions=sessions,
    tickers=TICKERS,
    batch_size=BATCH_SIZE,
    seed=17,
    mode="inspect",
    max_windows=BATCH_SIZE,
    shuffle=False,
)

batch = next(iter(dataset))
print("batch keys:", sorted(batch.keys()))
for key, value in batch.items():
    if torch.is_tensor(value):
        finite = bool(torch.isfinite(value).all())
        min_value = float(value.min()) if value.numel() else math.nan
        max_value = float(value.max()) if value.numel() else math.nan
        print(f"{key:20s} shape={tuple(value.shape)} dtype={value.dtype} finite={finite} min={min_value:.6g} max={max_value:.6g}")
    else:
        print(f"{key:20s} type={type(value).__name__} len={len(value) if hasattr(value, '__len__') else 'n/a'} sample={value[:5] if isinstance(value, list) else value}")

In [ ]:
batch["values"]

In [ ]:
# Inspect the first sample's latest context bar after input normalization.
feature_names = list(data_config.input_feature_columns)
time_feature_names = list(data_config.time_feature_columns)
last_values = batch["values"][0, -1].numpy()
last_time_features = batch["time_features"][0, -1].numpy()

print("first sample ticker:", batch.get("ticker", [None])[0])
print("current_close:", float(batch["current_close"][0]))
print("target_center:", float(batch["target_center"][0]), "target_scale:", float(batch["target_scale"][0]))

pl.DataFrame({"feature": feature_names, "normalized_value_at_context_end": last_values}).head(20)

In [ ]:
model = FeatureTemporalTransformer(
    feature_count=len(data_config.input_feature_columns),
    time_feature_count=len(data_config.time_feature_columns),
    context_length=data_config.context_length,
    horizon=data_config.horizon,
    target_count=len(data_config.target_columns),
    config=model_config,
).to(device)

if checkpoint is not None:
    state_dict = checkpoint.get("model", checkpoint)
    model.load_state_dict(state_dict, strict=True)
    print(f"loaded checkpoint: {checkpoint_path}")
    print("checkpoint step:", checkpoint.get("step"), "best_score:", checkpoint.get("best_score"))
else:
    print("CHECKPOINT_PATH is empty; using fresh randomly initialized model.")

param_count = sum(parameter.numel() for parameter in model.parameters())
print(f"model parameters={param_count:,}")

In [ ]:
# Keras-style model summary and graph view.
# Optional packages for richer output:
#   pip install torchinfo torchview graphviz

import inspect
from IPython.display import Markdown, display

SUMMARY_BATCH_SIZE = 1
SUMMARY_DEPTH = 4
GRAPH_DEPTH = 3


def _active_data_config():
    if "data_config" in globals():
        return data_config
    if "config" in globals() and hasattr(config, "data"):
        return config.data
    raise NameError("Run the config/model setup cells before this summary cell.")


def _model_device(model_obj):
    try:
        return next(model_obj.parameters()).device
    except StopIteration:
        return torch.device("cpu")


def _shape_of(value):
    if torch.is_tensor(value):
        return tuple(value.shape)
    if isinstance(value, (list, tuple)):
        return [_shape_of(item) for item in value]
    if isinstance(value, dict):
        return {key: _shape_of(item) for key, item in value.items()}
    return type(value).__name__


def _slice_input_tensor(value, rows, device):
    if torch.is_tensor(value):
        return value[:rows].to(device)
    return torch.as_tensor(value[:rows]).to(device)


def _branch_time_feature_count(branch):
    projection = getattr(branch, "time_projection", None)
    if projection is not None:
        for module in projection.modules():
            if isinstance(module, torch.nn.Linear):
                return int(module.in_features)
    data_cfg = _active_data_config()
    return len(getattr(data_cfg, "time_feature_columns", ()))


def _synthetic_input_for(name, rows, device):
    data_cfg = _active_data_config()
    dtype = torch.float32

    if name == "values":
        if hasattr(model, "feature_count"):
            return torch.zeros(rows, model.context_length, model.feature_count, dtype=dtype, device=device)
        return torch.zeros(
            rows,
            model.one_min_encoder.context_length,
            model.one_min_encoder.feature_count,
            dtype=dtype,
            device=device,
        )
    if name == "time_features":
        if hasattr(model, "time_projection"):
            time_count = len(getattr(data_cfg, "time_feature_columns", ()))
            return torch.zeros(rows, model.context_length, time_count, dtype=dtype, device=device)
        return torch.zeros(
            rows,
            model.one_min_encoder.context_length,
            _branch_time_feature_count(model.one_min_encoder),
            dtype=dtype,
            device=device,
        )

    branch_by_name = {
        "five_min_values": "five_min_encoder",
        "five_min_time_features": "five_min_encoder",
        "thirty_min_values": "thirty_min_encoder",
        "thirty_min_time_features": "thirty_min_encoder",
        "anchor_values": "anchor_encoder",
        "anchor_time_features": "anchor_encoder",
    }
    if name in branch_by_name:
        branch = getattr(model, branch_by_name[name])
        if name.endswith("time_features"):
            feature_count = _branch_time_feature_count(branch)
        else:
            feature_count = branch.feature_count
        return torch.zeros(rows, branch.context_length, feature_count, dtype=dtype, device=device)

    raise ValueError(f"Do not know how to build summary input for forward argument {name!r}.")


def _summary_inputs():
    device_for_summary = _model_device(model)
    forward_names = [
        name
        for name, parameter in inspect.signature(model.forward).parameters.items()
        if name != "self" and parameter.kind in (parameter.POSITIONAL_OR_KEYWORD, parameter.KEYWORD_ONLY)
    ]
    rows = SUMMARY_BATCH_SIZE
    if "batch" in globals() and all(name in batch for name in forward_names):
        rows = min(SUMMARY_BATCH_SIZE, int(batch[forward_names[0]].shape[0]))
        return tuple(_slice_input_tensor(batch[name], rows, device_for_summary) for name in forward_names)
    return tuple(_synthetic_input_for(name, rows, device_for_summary) for name in forward_names)


def _fallback_summary(model_obj, inputs):
    rows = []
    hooks = []
    module_names = {module: name for name, module in model_obj.named_modules()}

    def register_hook(module):
        if module is model_obj or list(module.children()):
            return

        def hook(mod, mod_inputs, mod_outputs):
            params = sum(parameter.numel() for parameter in mod.parameters(recurse=False))
            trainable = sum(parameter.numel() for parameter in mod.parameters(recurse=False) if parameter.requires_grad)
            rows.append(
                {
                    "layer": module_names.get(mod, ""),
                    "type": mod.__class__.__name__,
                    "input_shape": _shape_of(mod_inputs),
                    "output_shape": _shape_of(mod_outputs),
                    "params": params,
                    "trainable": trainable,
                }
            )

        hooks.append(module.register_forward_hook(hook))

    for module in model_obj.modules():
        register_hook(module)

    was_training = model_obj.training
    model_obj.eval()
    with torch.inference_mode():
        outputs = model_obj(*inputs)
    for hook in hooks:
        hook.remove()
    model_obj.train(was_training)

    print(f"Output shape: {_shape_of(outputs)}")
    total_params = sum(parameter.numel() for parameter in model_obj.parameters())
    trainable_params = sum(parameter.numel() for parameter in model_obj.parameters() if parameter.requires_grad)
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")
    try:
        import pandas as pd

        display(pd.DataFrame(rows))
    except ModuleNotFoundError:
        for row in rows:
            print(row)


summary_inputs = _summary_inputs()
input_shapes = [_shape_of(tensor) for tensor in summary_inputs]
display(Markdown(f"### {MODEL_VERSION} Model Summary"))
print("Forward input shapes:", input_shapes)

try:
    from torchinfo import summary

    summary_result = summary(
        model,
        input_data=summary_inputs,
        depth=SUMMARY_DEPTH,
        col_names=("input_size", "output_size", "num_params", "trainable"),
        verbose=0,
        device=str(_model_device(model)),
    )
    print(summary_result)
except ModuleNotFoundError:
    print("torchinfo is not installed; using the local hook-based summary fallback.")
    _fallback_summary(model, summary_inputs)
except Exception as exc:
    print(f"torchinfo summary failed: {exc}")
    print("Using the local hook-based summary fallback.")
    _fallback_summary(model, summary_inputs)

try:
    from torchview import draw_graph

    graph = draw_graph(
        model,
        input_data=summary_inputs,
        graph_name=f"{MODEL_VERSION}_model",
        depth=GRAPH_DEPTH,
        expand_nested=False,
        save_graph=False,
        device=str(_model_device(model)),
    )
    display(graph.visual_graph)
except ModuleNotFoundError:
    print("torchview is not installed, so the graph view is skipped. Install torchview and graphviz for a plot_model-style graph.")
except Exception as exc:
    print(f"torchview graph failed: {exc}")


In [ ]:
def move_tensor_batch(batch, device):
    return {key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value for key, value in batch.items()}

model.eval()
device_batch = move_tensor_batch(batch, device)
with torch.inference_mode():
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP and device.type == "cuda"):
        prediction, direction_logits = model(device_batch["values"], device_batch["time_features"])
        loss, loss_parts = forecast_loss(
            prediction,
            device_batch["targets"],
            direction_logits,
            device_batch["direction"],
            model_config.direction_loss_weight,
        )

print("prediction shape:", tuple(prediction.shape))
print("direction_logits shape:", tuple(direction_logits.shape))
print("loss:", float(loss.detach().cpu()))
print("loss_parts:", loss_parts)
print("prediction finite:", bool(torch.isfinite(prediction).all()))
print("target finite:", bool(torch.isfinite(device_batch["targets"]).all()))

In [ ]:
def prediction_and_target_prices(prediction_tensor, batch_dict, data_config):
    pred_np = prediction_tensor.detach().cpu().numpy()
    target_np = batch_dict["targets"].detach().cpu().numpy()
    current_close = batch_dict["current_close"].detach().cpu().numpy()
    if data_config.target_mode == "actual_price_zscore":
        center = batch_dict["target_center"].detach().cpu().numpy().reshape(-1, 1, 1)
        scale = batch_dict["target_scale"].detach().cpu().numpy().reshape(-1, 1, 1)
        return pred_np * scale + center, target_np * scale + center
    if data_config.target_mode == "return_bps":
        current = np.maximum(current_close.reshape(-1, 1, 1), 1e-6)
        return current * np.exp(pred_np / 10000.0), current * np.exp(target_np / 10000.0)
    raise ValueError(data_config.target_mode)

pred_prices, target_prices = prediction_and_target_prices(prediction, device_batch, data_config)
pred_bps = target_values_to_bps(
    prediction.detach().cpu().numpy(),
    batch["current_close"].numpy(),
    batch["target_center"].numpy(),
    batch["target_scale"].numpy(),
    data_config.target_mode,
)
target_bps = target_values_to_bps(
    batch["targets"].numpy(),
    batch["current_close"].numpy(),
    batch["target_center"].numpy(),
    batch["target_scale"].numpy(),
    data_config.target_mode,
)

close_index = data_config.target_columns.index("close")
rows = []
row_count = min(len(batch.get("ticker", [])), pred_bps.shape[0], 20)
for idx in range(row_count):
    rows.append({
        "row": idx,
        "ticker": batch.get("ticker", [None] * BATCH_SIZE)[idx],
        "target_timestamp_ns": int(batch["target_timestamp_ns"][idx]),
        "current_close": float(batch["current_close"][idx]),
        "target_close": float(target_prices[idx, 0, close_index]),
        "prediction_close": float(pred_prices[idx, 0, close_index]),
        "target_bps": float(target_bps[idx, 0, close_index]),
        "prediction_bps": float(pred_bps[idx, 0, close_index]),
        "abs_error_bps": abs(float(pred_bps[idx, 0, close_index] - target_bps[idx, 0, close_index])),
        "dir_correct": bool((pred_bps[idx, 0, close_index] > 0.0) == (target_bps[idx, 0, close_index] > 0.0)),
    })

pl.DataFrame(rows)